In [1]:
!pip install -U langchain langchain-core langchain-community transformers huggingface_hub

  Using cached huggingface_hub-1.10.2-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.10.2-py3-none-any.whl (642 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.18.0
    Uninstalling huggingface-hub-0.18.0:
      Successfully uninstalled huggingface-hub-0.18.0


In [2]:
# Downgrade huggingface_hub to a compatible version
!pip install 'huggingface_hub<0.19'

  Using cached huggingface_hub-0.18.0-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.18.0-py3-none-any.whl (301 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.2
    Uninstalling huggingface_hub-1.10.2:
      Successfully uninstalled huggingface_hub-1.10.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.5.4 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.18.0 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.18.0 which is incompatible.
sentence-transformers 5.4.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.18.0 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.18.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<

### **IMPORTANT: After running the cell above, you MUST restart your Colab runtime and then run all cells from the beginning.**

To restart the runtime, go to `Runtime` -> `Restart runtime` in the Colab menu. After restarting, ensure you run all cells in order from the top of the notebook.

In [3]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFaceHub

In [4]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_token_here"

In [5]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_key"

In [12]:
llm = HuggingFaceHub(
    repo_id="google/flan-t5-base",
    model_kwargs={"temperature": 0.5},
    task="text2text-generation" # Add the task parameter
)

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract the following from the resume:

- Skills
- Tools
- Experience

Rules:
- Do NOT assume anything not present
- Return in structured JSON format

Resume:
{resume}
"""
)

scoring_prompt = PromptTemplate(
    input_variables=["resume_data", "job"],
    template="""
Compare resume data with the job description.

Resume Data:
{resume_data}

Job Description:
{job}

Give:
1. Match Score (0-100)
2. Matching Skills
3. Missing Skills
4. Explanation

Rules:
- Do NOT hallucinate
- Be strict in evaluation
- Return in structured JSON format
"""
)

extraction_chain = extraction_prompt | llm | JsonOutputParser()
scoring_chain = scoring_prompt | llm | JsonOutputParser()

In [8]:
strong_resume = "Python, Machine Learning, Deep Learning, SQL, AWS, 3 years experience"
average_resume = "Python, SQL, basic ML, 1 year experience"
weak_resume = "Excel, basic programming"

job_description = "Looking for Data Scientist with Python, ML, Deep Learning, AWS"

In [9]:
def run_pipeline(resume):
    extracted = extraction_chain.invoke({"resume": resume})

    result = scoring_chain.invoke({
        "resume_data": extracted,
        "job": job_description
    })

    return result

In [ ]:
print("🔹 STRONG CANDIDATE:\n", run_pipeline(strong_resume))

print("\n🔹 AVERAGE CANDIDATE:\n", run_pipeline(average_resume))

print("\n🔹 WEAK CANDIDATE:\n", run_pipeline(weak_resume))